## Modelado.


La variable objetivo del modelo es log_ratio, definida como el logaritmo del ratio de viajeros, lo que permite estabilizar la varianza y mejorar el ajuste de los modelos.

In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import json
from sklearn.model_selection import TimeSeriesSplit

In [2]:
train_viajeros = pd.read_csv("train_viajeros.csv")
df_verano_viajeros = pd.read_csv("verano_viajeros.csv")
df_viajeros_totales = pd.read_csv("viajeros_totales.csv")

Se quita el mes de mayo del conjunto de entrenamiento. Solo se usa para crear la variable lag_1 para predecir el mes de junio.

In [3]:
train_viajeros = train_viajeros[train_viajeros["Mes"].isin([6,7,8])].copy()

In [4]:
train_viajeros = (train_viajeros.sort_values(["Año", "Mes"]).reset_index(drop=True))

Definición de variables.

In [5]:
x_columnas = ["Provincias", "Año", "Mes", "log_lag_estacional", "log_lag_1", "log_lag_2", "log_lag_3", "log_lag_4", "media_lag_2", "media_lag_3", "tendencia_corta"] 
y_columnas = "log_ratio"

categoricas_viajeros = [x_columnas.index("Provincias")]

### Modelo base: Naive

In [6]:
datos_2024 = train_viajeros[train_viajeros["Año"] == 2024]
true_naive = datos_2024[y_columnas]

prediccion_naive = np.zeros(len(true_naive))

mae_naive = mean_absolute_error(true_naive, prediccion_naive)
rmse_naive = np.sqrt(mean_squared_error(true_naive, prediccion_naive))

print("--- MODELO NAIVE ---")
print("MAE:", mae_naive)
print("RMSE:", rmse_naive)

--- MODELO NAIVE ---
MAE: 0.165972487826781
RMSE: 0.22103603168146027


## Algoritmos candidatos

### 1. CatBoost

Selección de hiperparámetros: Grid Search con TimeSeriesSplit.

In [7]:

tscv = TimeSeriesSplit(n_splits=3)

depth_values = [4, 5, 6]
learning_rates = [0.02, 0.03, 0.04]
l2_values = [3, 4, 5, 6]

resultados_viajeros = []

for depth in depth_values:
    for learning in learning_rates:
        for l2 in l2_values:

            cv_rmses = []

            cv_maes = []

            for train_index, validacion_index in tscv.split(train_viajeros):

                x_train_fold = train_viajeros.iloc[train_index][x_columnas]
                y_train_fold = train_viajeros.iloc[train_index][y_columnas]

                x_validacion_fold = train_viajeros.iloc[validacion_index][x_columnas]
                y_validacion_fold = train_viajeros.iloc[validacion_index][y_columnas]

                modelo = CatBoostRegressor(
                    iterations = 700,
                    learning_rate = learning,
                    depth = depth,
                    l2_leaf_reg = l2,
                    loss_function = "RMSE",
                    random_seed = 42,
                    early_stopping_rounds = 50,
                    verbose=0
                )

                modelo.fit(x_train_fold, y_train_fold, cat_features=categoricas_viajeros, eval_set=(x_validacion_fold, y_validacion_fold))

                prediccion = modelo.predict(x_validacion_fold)

                cv_rmses.append(np.sqrt(mean_squared_error(y_validacion_fold, prediccion)))
                cv_maes.append(mean_absolute_error(y_validacion_fold, prediccion))

            avg_rmse = np.mean(cv_rmses)
            avg_mae = np.mean(cv_maes)
            
            print(f"Depth: {depth}, Learning_rate: {learning}, l2: {l2}")

            resultados_viajeros.append({"depth" : depth, "learning_rate" : learning, "l2" : l2, "MAE" : avg_mae, "RMSE" : avg_rmse})


Depth: 4, Learning_rate: 0.02, l2: 3
Depth: 4, Learning_rate: 0.02, l2: 4
Depth: 4, Learning_rate: 0.02, l2: 5
Depth: 4, Learning_rate: 0.02, l2: 6
Depth: 4, Learning_rate: 0.03, l2: 3
Depth: 4, Learning_rate: 0.03, l2: 4
Depth: 4, Learning_rate: 0.03, l2: 5
Depth: 4, Learning_rate: 0.03, l2: 6
Depth: 4, Learning_rate: 0.04, l2: 3
Depth: 4, Learning_rate: 0.04, l2: 4
Depth: 4, Learning_rate: 0.04, l2: 5
Depth: 4, Learning_rate: 0.04, l2: 6
Depth: 5, Learning_rate: 0.02, l2: 3
Depth: 5, Learning_rate: 0.02, l2: 4
Depth: 5, Learning_rate: 0.02, l2: 5
Depth: 5, Learning_rate: 0.02, l2: 6
Depth: 5, Learning_rate: 0.03, l2: 3
Depth: 5, Learning_rate: 0.03, l2: 4
Depth: 5, Learning_rate: 0.03, l2: 5
Depth: 5, Learning_rate: 0.03, l2: 6
Depth: 5, Learning_rate: 0.04, l2: 3
Depth: 5, Learning_rate: 0.04, l2: 4
Depth: 5, Learning_rate: 0.04, l2: 5
Depth: 5, Learning_rate: 0.04, l2: 6
Depth: 6, Learning_rate: 0.02, l2: 3
Depth: 6, Learning_rate: 0.02, l2: 4
Depth: 6, Learning_rate: 0.02, l2: 5
D

Selección de los mejores parámetros.

In [8]:
df_resultados_viajeros = pd.DataFrame(resultados_viajeros)
df_resultados_viajeros = df_resultados_viajeros.sort_values("RMSE")

mejores_parametros = df_resultados_viajeros.iloc[0]
mejor_rmse = mejores_parametros["RMSE"]

diferecnia = rmse_naive - mejores_parametros["RMSE"]

if mejores_parametros["RMSE"] < rmse_naive:
    print(f"El modelo mejora al Naive")
else:
    print("El modelo no mejora al Naive")

parametros = {"depth" : int(mejores_parametros["depth"]), "learning_rate" : float(mejores_parametros["learning_rate"]), "l2" : int(mejores_parametros["l2"])}

print("Mejores parámetros:")
print(mejores_parametros)

El modelo mejora al Naive
Mejores parámetros:
depth            6.000000
learning_rate    0.030000
l2               3.000000
MAE              0.117266
RMSE             0.157509
Name: 28, dtype: float64


In [9]:
archivo = "parámetros_columnas.json"

datos = {
    "mejores_parametros" : parametros,
    "x_columnas" : x_columnas,
    "y_columnas" : y_columnas
}

with open (archivo, "w") as f:
    json.dump(datos, f, indent=4)


### Otros modelos candidatos (comparativa)

En esta sección comparamos varios algoritmos usando el ismo esquema de validación temporal (`TimeSeriesSplit`) y las mismas métricas (MAE y RMSE) Para modelos que no aceptan variables categóricas directamente, aplicamos One-Hot Encoding solo sobre `Provincias`.


### 2. XGBoost Regressor + Random search

In [17]:
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error



In [11]:
X = train_viajeros[x_columnas]
y = train_viajeros[y_columnas]

Preprocesado (One-Hot solo para Provincias)

In [12]:
preprocess = ColumnTransformer(
    transformers=[
        ("prov", OneHotEncoder(handle_unknown="ignore"), ["Provincias"])
    ],
    remainder="passthrough"
)

 XGBoost (sin fijar hiperparámetros “finales” aun

In [13]:
xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

pipe_xgb = Pipeline(steps=[
    ("prep", preprocess),
    ("model", xgb)
])

# Validación temporal 
tscv = TimeSeriesSplit(n_splits=3)

Espacio de búsqueda (distribuciones / listas)

In [14]:
param_distributions = {
    "model__n_estimators": [300, 600, 900, 1200],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__reg_alpha": [0.0, 0.1, 1.0],
    "model__reg_lambda": [1.0, 3.0, 10.0]
}

In [18]:
search_xgb = RandomizedSearchCV(
    estimator=pipe_xgb,
    param_distributions=param_distributions,
    n_iter=25,  # si os da tiempo: 40-60
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search_xgb.fit(X, y)

print("Mejor RMSE (CV):", -search_xgb.best_score_)
print("Mejores hiperparámetros:", search_xgb.best_params_)

best_xgb = search_xgb.best_estimator_

Fitting 3 folds for each of 25 candidates, totalling 75 fits
Mejor RMSE (CV): 0.161213478061878
Mejores hiperparámetros: {'model__subsample': 1.0, 'model__reg_lambda': 1.0, 'model__reg_alpha': 0.0, 'model__n_estimators': 300, 'model__min_child_weight': 1, 'model__max_depth': 8, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.8}


Calculo de MAE y RMSE en CV con el mejor modelo

In [19]:
rmse_cv, mae_cv = [], []

for train_idx, val_idx in tscv.split(X):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    best_xgb.fit(X_tr, y_tr)
    pred = best_xgb.predict(X_val)

    rmse_cv.append(np.sqrt(mean_squared_error(y_val, pred)))
    mae_cv.append(mean_absolute_error(y_val, pred))

rmse_xgb = float(np.mean(rmse_cv))
mae_xgb = float(np.mean(mae_cv))

print("\nXGBoost (mejor config) - CV")
print("MAE:", mae_xgb)
print("RMSE:", rmse_xgb)


XGBoost (mejor config) - CV
MAE: 0.11885423520585174
RMSE: 0.161213478061878


## COMPARATIVAS

### Comparación directa de XGBoost Regressor  con CatBoost y Naive

In [20]:
comparativa = pd.DataFrame([
    {"Modelo": "Naive", "MAE": mae_naive, "RMSE": rmse_naive},
    {"Modelo": "CatBoost", "MAE": mejores_parametros["MAE"], "RMSE": mejores_parametros["RMSE"]},
    {"Modelo": "XGBoost", "MAE": mae_xgb, "RMSE": rmse_xgb}
]).sort_values("RMSE")

comparativa


,Modelo,MAE,RMSE
1,CatBoost,0.117266,0.157509
2,XGBoost,0.118854,0.161213
0,Naive,0.165972,0.221036


Conclusiones parciales:

Tras la optimización de los modelos candidatos, los resultados parciales muestran que tanto CatBoost como XGBoost mejoran de forma significativa al modelo base. Sin embargo, CatBoost Regressor obtiene el mejor rendimiento, con un RMSE de 0.1575 frente a 0.1612 de XGBoost, incluso después de optimizar este último mediante Random Search con validación temporal.
Además de su mejor desempeño, CatBoost presenta la ventaja adicional de tratar variables categóricas de forma nativa, lo que simplifica el preprocesamiento y reduce la complejidad del pipeline. Por estos motivos, se selecciona CatBoost Regressor como modelo final del proyecto.